[DeepSeek tool call][1]

`bash` 由这个 notebook 定义并在本机执行。DeepSeek 只返回要运行的 `command`。

<!-- Reference -->
[1]: https://api-docs.deepseek.com/zh-cn/guides/tool_calls/

In [4]:
# 1. Load library dependencies
from openai import OpenAI
from dotenv import load_dotenv
import json
import os
import subprocess

In [5]:
# 2. load environment variable from .env file

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

In [6]:
client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
)


def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=messages,
        tools=tools,
        extra_body={"thinking": {"type": "disabled"}},
    )
    return response.choices[0].message


WORKDIR = os.getcwd()
TIMEOUT_SECONDS = 5
MAX_OUTPUT_CHARS = 4000


def trim(text):
    if len(text) <= MAX_OUTPUT_CHARS:
        return text
    return text[:MAX_OUTPUT_CHARS] + "\n...[truncated]"


def run_bash(command):
    command = (command or "").strip()
    if not command:
        return "exit_code=2\nerror: empty command"
    try:
        completed = subprocess.run(
            ["bash", "-c", command],
            cwd=WORKDIR,
            capture_output=True,
            text=True,
            timeout=TIMEOUT_SECONDS,
        )
    except subprocess.TimeoutExpired as exc:
        partial = (exc.stdout or "") + (exc.stderr or "")
        return trim(
            f"exit_code=124\nerror: timed out after {TIMEOUT_SECONDS}s\n{partial}"
        )
    body = ((completed.stdout or "") + (completed.stderr or "")).rstrip()
    if not body:
        body = "(no output)"
    return trim(f"exit_code={completed.returncode}\n{body}")


tools = [
    {
        "type": "function",
        "function": {
            "name": "bash",
            "description": (
                "Run one bash command in the notebook working directory "
                "and return stdout, stderr, and the exit code."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {
                        "type": "string",
                        "description": "One bash command, for example: date '+%Y-%m-%d %H:%M:%S %Z'",
                    },
                },
                "required": ["command"],
            },
        },
    },
]

messages = [
    {
        "role": "user",
        "content": (
            "Use the bash tool to run `date '+%Y-%m-%d %H:%M:%S %Z'` "
            "and quote the output."
        ),
    }
]

In [7]:
message = send_messages(messages)
print(f"User>\t {messages[0]['content']}")

User>	 Use the bash tool to run `date '+%Y-%m-%d %H:%M:%S %Z'` and quote the output.


In [8]:
message.model_dump()

{'content': '',
 'refusal': None,
 'role': 'assistant',
 'annotations': None,
 'audio': None,
 'function_call': None,
 'tool_calls': [{'id': 'call_00_3SuyP2C369N5z9SafdDn1496',
   'function': {'arguments': '{"command": "date \'+%Y-%m-%d %H:%M:%S %Z\'"}',
    'name': 'bash'},
   'type': 'function',
   'index': 0}]}

In [9]:
message.tool_calls[0].model_dump()

{'id': 'call_00_3SuyP2C369N5z9SafdDn1496',
 'function': {'arguments': '{"command": "date \'+%Y-%m-%d %H:%M:%S %Z\'"}',
  'name': 'bash'},
 'type': 'function',
 'index': 0}

In [10]:
# The model only proposes the command. This cell runs it locally.
rounds = 0
while message.tool_calls and rounds < 3:
    messages.append(message)
    for call in message.tool_calls:
        name = call.function.name
        try:
            args = json.loads(call.function.arguments or "{}")
        except json.JSONDecodeError as exc:
            result = f"error: invalid arguments: {exc}"
        else:
            if name != "bash":
                result = f"unknown tool: {name}"
            else:
                command = args.get("command", "")
                print(f"bash>\t {command}")
                result = run_bash(command)
                print(result)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": call.id,
                "content": result,
            }
        )
    message = send_messages(messages)
    rounds += 1

print(f"Model>\t {message.content}")

bash>	 date '+%Y-%m-%d %H:%M:%S %Z'
exit_code=0
2026-11-07 11:19:03 CST
Model>	 The command output was:

`2026-11-07 11:19:03 CST`


In [11]:
message.model_dump()

{'content': 'The command output was:\n\n`2026-11-07 11:19:03 CST`',
 'refusal': None,
 'role': 'assistant',
 'annotations': None,
 'audio': None,
 'function_call': None,
 'tool_calls': None}